# Power Plant · четыре модели на одной GPU

MLP, MGS-KAN, Deep MLP и ResNet обучаются на четырёх признаках Power Plant, фолд 0. Для каждой пары seed используются одинаковые разбиение и порядок мини-батчей. Всего 12 обучений по 100 эпох. Оценка по валидации, чекпоинты сохраняются в MLflow.

Выбери GPU, доступную ядру Jupyter. Проверь в терминале загрузку через `nvidia-smi`. Нужны файл `data/power_plant.csv`, доступ к MLflow из корневого `.env` и видимость памяти процессов в `nvidia-smi`.

In [ ]:
PROJECT_DIR = None
DEVICE = "cuda:0"
MODES = (4,)  # Для сравнения на одинаковых заданиях: (4, 1, 2, 3).
VERSION = "v1"  # Для чистого повторного замера задай новую версию.
CPU_THREADS_TOTAL = 4
GPU_BUDGET_MIB = 4096

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

In [ ]:
from time import monotonic

import pandas as pd
from IPython.display import display
from src.config import ModelSettings, MonitorSettings, ResourceSettings, TrackingSettings, TrainingSettings
from src.experiments.search import SearchStage, ValidationGrid

models = ["mlp", "mgs_kan", "deep_mlp", "resnet"]
seeds = [(11, 101), (22, 202), (33, 303)]
# Архитектуры Power Plant, выбранные на валидации v1.
architecture = ModelSettings(mlp_width=12, single_kernels=4, pair_kernels=2, deep_width=128, deep_depth=4)
training = TrainingSettings(epochs=100, batch_size=64, learning_rate=0.001)
grid = ValidationGrid(
    datasets=("power_plant",), folds=(0,), seeds=seeds, training=training,
    stages=(SearchStage(name="selected-v1", models=models, settings=architecture),),
)
tracking = TrackingSettings(_env_file=ENV_FILE)
monitoring = MonitorSettings(_env_file=ENV_FILE)
if not DEVICE.startswith("cuda:") or not DEVICE[5:].isdecimal():
    raise ValueError("Укажи индекс GPU, например cuda:0")
if not MODES or len(set(MODES)) != len(MODES) or any(slots not in (1, 2, 3, 4) for slots in MODES):
    raise ValueError("Укажи разные режимы от 1 до 4")
if CPU_THREADS_TOTAL < max(MODES):
    raise ValueError("Нужен хотя бы один поток CPU на процесс")

In [ ]:
timings = []
scores = []
for slots in MODES:
    resources = ResourceSettings(
        _env_file=ENV_FILE, max_parallel_trials=slots, cpu_threads=CPU_THREADS_TOTAL // slots,
        gpu_budget_mib=GPU_BUDGET_MIB,
    )
    started = monotonic()
    runs = grid.run(
        version=f"throughput-{VERSION}-p{slots}", device=DEVICE, root=DATA_DIR,
        tracking=tracking, monitoring=monitoring, resources=resources,
    )
    seconds = monotonic() - started
    timings.append({
        "parallel": slots, "elapsed_s": round(seconds, 1), "cached": runs.attrs["reused"],
        "runs_per_hour": len(runs) * 3600 / seconds if runs.attrs["reused"] == 0 else None,
        "peak_workers_MiB": runs.attrs["peak_gpu_mib"],
    })
    scores.append(runs.assign(parallel=slots))
display(pd.DataFrame(timings))
display(pd.concat(scores).pivot(index=["model", "init_seed", "shuffle_seed"], columns="parallel", values="val_rmse"))

Для возобновления оставь прежние `VERSION` и число процессов. MLflow переиспользует готовые запуски. Скорость рассчитывается только для полностью новых серий. Пик памяти — сумма по процессам серии в момент замера. Для следующего эксперимента выбирай режим по числу обучений в час при сопоставимом RMSE и бюджете памяти.